# BloomBench — Extended Benchmark (HPO pass)

One-time hyperparameter search across all benchmark datasets. Calls into the library module [`pysephone.benchmarks.bloombench`](../src/pysephone/benchmarks/bloombench/) so this notebook is just orchestration. Outputs land in `outputs/bloombench/hyperparams/agera5/<dataset>_<model>.json` (best params) and `..._trials.json` (full per-trial log).

**This notebook is meant to be run ONCE.** Subsequent replications use the eval notebook ([`bloombench_extended.ipynb`](bloombench_extended.ipynb)), which reads the cached params via the same module.

**Search strategy**:
* **Trees (RF, XGBoost)** — `RandomizedSearchCV` with year-aware `GroupKFold(5)`, 20 sampled configs per (dataset, model). Year-disjoint CV so future years don't leak into past-year folds during the search.
* **Torch (CNN1D, LSTM, Transformer)** — plain random search (15 trials). Each trial fits on `train' = first 80% of training years` and is scored by MAE on `val_cutoff = last 20% of training years`. `fit()`'s built-in random year val handles early stopping; the external cutoff val handles HPO selection. Two non-overlapping val sets.

**Why HPO is keyed by `(dataset, model)` only — no seed.** HPO findings shouldn't depend on the fit seed. When the eval notebook runs seeds 1..N it reuses these cached params for free; only the *fitting* step is reseeded.

**Expected wall clock**: tree HPO ~2–4 h, torch HPO ~20–40 h on this 18-dataset list. Plan for an overnight run.

Climate provider: default is AgERA5. Run [`download_agera5.ipynb`](download_agera5.ipynb) once to populate the HDF5 cache before invoking HPO.

## 1. Setup

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pysephone.benchmarks.bloombench import (
    config as bb_config,
    load_bloombench_datasets,
    load_hp_cache,
    load_hp_trials,
    run_hpo,
)

# HPO is single-seed by convention. The eval notebook re-uses cached params
# over many seeds without re-tuning.
HPO_SEED = 0

# Force-retune list. False == skip pairs that already have a cached best_params.json.
FORCE_RETUNE = False

print(f'Datasets requested:  {len(bb_config.DATASETS_REQUESTED)}')
print(f'Min dataset samples: {bb_config.MIN_DATASET_SAMPLES}')
print(f'Features:            {bb_config.FEATURE_KEYS}')
print(f'Train fraction:      {bb_config.SPLIT_SIZE}')
print(f'HPO seed:            {HPO_SEED}')
print(f'Torch device:        {bb_config.torch_device()}')
print(f'Tree n_iter / folds: {bb_config.HPO_N_ITER_TREES} / {bb_config.HPO_CV_FOLDS}')
print(f'Torch trials:        {bb_config.HPO_N_TRIALS_TORCH}')
print(f'Force retune:        {FORCE_RETUNE}')

## 2. Existing HP cache snapshot

Lists which (dataset, model) pairs already have a cached `best_params.json` from a prior HPO run. The run loop below will *skip* those pairs unless they appear in `FORCE_RETUNE_MODELS` — so this snapshot is how you confirm what work remains.

In [ ]:
TUNED_MODELS = ['RandomForest', 'XGBoost', 'CNN1D', 'LSTM', 'Transformer']
rows = []
for ds_name, _ in bb_config.DATASETS_REQUESTED:
    for model_key in TUNED_MODELS:
        cached = load_hp_cache(ds_name, model_key)
        rows.append(dict(
            dataset=ds_name, model=model_key,
            cached=(cached is not None),
            params='' if cached is None else ', '.join(f'{k}={v}' for k, v in sorted(cached.items())),
        ))
snapshot = pd.DataFrame(rows)
n_hits = snapshot['cached'].sum()
print(f'HP cache:  {n_hits} / {len(snapshot)} pairs already tuned.')
snapshot

## 3. Load datasets

Same as in the eval notebook — NPN entries with <100 samples are dropped at load time.

In [ ]:
datasets, load_summary = load_bloombench_datasets(verbose=True)
pd.DataFrame(load_summary)

## 4. Run HPO — tune every tunable model on every dataset

`run_hpo` skips pairs that already have a cached `best_params.json` (unless `FORCE_RETUNE=True`). Mean / Linear have no hyperparameters and are skipped silently.

In [ ]:
hpo_rows = run_hpo(
    datasets,
    models=TUNED_MODELS,
    seed=HPO_SEED,
    force_retune=FORCE_RETUNE,
    verbose=True,
)
pd.DataFrame(hpo_rows)

## 5. Final HP cache table

Shows the cached `best_params.json` for every (dataset, model) pair after the run loop completed. This is the artifact the eval notebook consumes.

In [ ]:
rows = []
for ds_name, _ in bb_config.DATASETS_REQUESTED:
    if ds_name not in datasets:
        continue
    for model_key in TUNED_MODELS:
        cached = load_hp_cache(ds_name, model_key)
        if cached is None:
            rows.append(dict(dataset=ds_name, model=model_key, params='(no cache)'))
        else:
            rows.append(dict(
                dataset=ds_name, model=model_key,
                params=', '.join(f'{k}={v}' for k, v in sorted(cached.items())),
            ))
pd.DataFrame(rows)

## 6. Torch random-search trial trajectories

Per (dataset, torch-model), shows val-MAE per trial with the selected best marked. Use these to sanity-check that the search converged — if val-MAE values are tightly bunched and the best is only marginally ahead, the cached config is on shakier ground.

In [ ]:
TORCH_TUNED = ['CNN1D', 'LSTM', 'Transformer']
pairs = [(ds, m) for ds, _ in bb_config.DATASETS_REQUESTED for m in TORCH_TUNED
         if ds in datasets and load_hp_trials(ds, m) is not None]

if not pairs:
    print('No torch trial logs yet — re-run section 4.')
else:
    n = len(pairs)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.0 * ncols, 3.2 * nrows), squeeze=False)
    for ax, (ds_name, model_key) in zip(axes.flat, pairs):
        trials_log = load_hp_trials(ds_name, model_key)
        ix = [t['trial'] for t in trials_log]
        val = [t['val_mae'] if np.isfinite(t['val_mae']) else np.nan for t in trials_log]
        ax.plot(ix, val, marker='o', linestyle='-', color='steelblue')
        finite = [v for v in val if np.isfinite(v)]
        if finite:
            best = min(finite)
            ax.axhline(best, color='crimson', linestyle='--', linewidth=1, label=f'best = {best:.2f}')
            ax.legend(loc='upper right', fontsize=9)
        ax.set_title(f'{ds_name} — {model_key}')
        ax.set_xlabel('Trial')
        ax.set_ylabel('Val MAE')
    for ax in axes.flat[len(pairs):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()

## Next steps

Once HPO finishes, the cache is populated. Switch to [`bloombench_extended.ipynb`](bloombench_extended.ipynb) for the evaluation pass — it reads the cached params via the same module and runs over your desired list of seeds without re-triggering HPO.